# GDA Phase 4B + Phase 4C: One-Click Reproduction Notebook

This notebook reproduces every numerical table in the paper *Measuring Alignment Friction: A CSV-Backed Gradient Decomposition Assay of Direct Constraint and Counterfactual Reframing in Frontier Models* (Gallegos, April 2026) directly from the released CSV files in the GitHub repo.

**What this notebook does:**
1. Downloads the released metrics CSVs from GitHub.
2. Verifies row counts and validity flags match what the paper reports.
3. Reproduces Table 1 (Phase 4B vector summary), Table 2 (Phase 4B AC vs FM by model), Tables 3-7 (Phase 4C results).
4. Outputs a side-by-side comparison of paper-reported values vs CSV-computed values to verify match.

**What this notebook does NOT do:**
- It does not re-run the assay. Re-running requires an OpenRouter API key and approximately $19-35 in API spend per phase.
- It does not compute inferential statistics. The paper reports descriptive means; significance tests are left to readers.
- It does not validate the evaluator. The single-evaluator design is the paper's most load-bearing methodological gap (Section 6).

**Runtime:** Approximately 30 seconds on a free Colab instance. No GPU required. No paid services required.

**Repo:** https://github.com/devinendorphin/alignment-friction-gda

## 1. Setup: Download the released CSVs

In [ ]:
import pandas as pd
import urllib.request
from pathlib import Path

REPO_RAW = "https://raw.githubusercontent.com/devinendorphin/alignment-friction-gda/main"

FILES_TO_DOWNLOAD = [
    # Phase 4B
    "04_GDA_Metrics_VALID_canonical.csv",
    "06_GDA_Vector_Summary_VALID.csv",
    "07_GDA_Model_Vector_Summary_VALID.csv",
    # Phase 4C
    "phase-4c/04_Phase4C_Metrics_VALID_canonical.csv",
    "phase-4c/06_Phase4C_Condition_Summary_VALID.csv",
    "phase-4c/07_Phase4C_Model_Condition_Summary_VALID.csv",
    "phase-4c/08_Phase4C_Void_Response_Categorical.csv",
]

Path("phase-4c").mkdir(exist_ok=True)

for f in FILES_TO_DOWNLOAD:
    out = Path(f)
    if out.exists():
        continue
    url = f"{REPO_RAW}/{f}"
    print(f"Downloading {f}...")
    urllib.request.urlretrieve(url, out)

print("\nAll files ready.")

## 2. Sanity check: row counts match what the paper reports

In [ ]:
df_4b = pd.read_csv("04_GDA_Metrics_VALID_canonical.csv")
df_4c = pd.read_csv("phase4c/04_Phase4C_Metrics_VALID_canonical.csv")

print(f"Phase 4B valid rows: {len(df_4b)}  (paper reports 1,984)")
print(f"Phase 4C valid rows: {len(df_4c)}  (paper reports 1,199)")

assert len(df_4b) == 1984, "Phase 4B row count mismatch"
assert len(df_4c) == 1199, "Phase 4C row count mismatch"
print("\n✓ Row counts match.")

## 3. Reproduce Phase 4B Table 1: Vector-level descriptive means

From the paper (Section 4.1):

In [ ]:
PAPER_TABLE_1 = pd.DataFrame({
    "Vector": ["Control_Ecology", "Constraint_Syntax", "Sensitive_Structural",
               "Conflict_Tradeoff", "Taboo_Asymmetry", "Self_Audit_Linguistic",
               "Adversarial_Compression", "Fictional_Mirror"],
    "Valid_n_paper":  [246, 249, 249, 250, 245, 248, 248, 249],
    "phi_c_paper":    [8.69, 7.31, 8.84, 8.77, 8.64, 5.18, 3.56, 9.01],
    "phi_s_paper":    [8.32, 6.65, 8.06, 8.94, 7.95, 5.05, 2.35, 8.83],
    "drag_paper":     [0.25, 0.05, 0.78, 0.85, 1.29, 1.75, 6.20, 0.65],
    "boil_paper":     [0.20, 0.04, 0.47, 0.38, 0.61, 2.50, 6.28, 0.17],
})

VECTOR_ORDER = PAPER_TABLE_1["Vector"].tolist()

agg = (df_4b.groupby("Vector")
       .agg(Valid_n_csv=("phi_content", "size"),
            phi_c_csv=("phi_content", "mean"),
            phi_s_csv=("phi_specificity", "mean"),
            drag_csv=("safety_drag", "mean"),
            boil_csv=("boilerplate_intensity", "mean"))
       .round(2)
       .reindex(VECTOR_ORDER)
       .reset_index())

comparison = PAPER_TABLE_1.merge(agg, on="Vector")
for col in ["phi_c", "phi_s", "drag", "boil"]:
    comparison[f"match_{col}"] = (
        (comparison[f"{col}_paper"] - comparison[f"{col}_csv"]).abs() < 0.01
    )
comparison["match_n"] = comparison["Valid_n_paper"] == comparison["Valid_n_csv"]

print("PHASE 4B TABLE 1: Paper vs CSV")
print(comparison[["Vector", "Valid_n_csv", "phi_c_csv", "phi_s_csv", "drag_csv", "boil_csv"]].to_string(index=False))
print()
all_match = comparison[["match_n", "match_phi_c", "match_phi_s", "match_drag", "match_boil"]].all().all()
print(f"\n{'✓' if all_match else '✗'} All Phase 4B Table 1 values {'match' if all_match else 'MISMATCH'} the paper.")

## 4. Reproduce Phase 4B Table 2: Adversarial_Compression vs Fictional_Mirror by model

From the paper (Section 4):

In [ ]:
PAPER_TABLE_2 = pd.DataFrame({
    "Model": ["anthropic/claude-opus-4.6", "google/gemini-3.1-pro-preview",
              "meta-llama/llama-3.3-70b-instruct", "openai/gpt-5.2", "x-ai/grok-4.20"],
    "AC_phi_c_paper":  [8.19, 1.86, 2.66, 1.86, 3.18],
    "AC_drag_paper":   [2.41, 6.92, 7.94, 5.81, 7.97],
    "FM_phi_c_paper":  [9.14, 9.14, 8.64, 9.04, 9.10],
    "FM_drag_paper":   [0.58, 0.46, 0.76, 0.66, 0.78],
})

ac_fm = (df_4b[df_4b["Vector"].isin(["Adversarial_Compression", "Fictional_Mirror"])]
         .groupby(["Model", "Vector"])
         .agg(phi_c=("phi_content", "mean"), drag=("safety_drag", "mean"))
         .round(2)
         .unstack("Vector"))
ac_fm.columns = [f"{v}_{m}" for m, v in ac_fm.columns]
ac_fm = ac_fm.reset_index()

merged = PAPER_TABLE_2.merge(
    ac_fm.rename(columns={
        "phi_c_Adversarial_Compression": "AC_phi_c_csv",
        "drag_Adversarial_Compression": "AC_drag_csv",
        "phi_c_Fictional_Mirror": "FM_phi_c_csv",
        "drag_Fictional_Mirror": "FM_drag_csv",
    }),
    on="Model",
)
for col in ["AC_phi_c", "AC_drag", "FM_phi_c", "FM_drag"]:
    merged[f"match_{col}"] = (merged[f"{col}_paper"] - merged[f"{col}_csv"]).abs() < 0.01

print("PHASE 4B TABLE 2: AC vs FM by model\n")
display_cols = ["Model", "AC_phi_c_csv", "AC_drag_csv", "FM_phi_c_csv", "FM_drag_csv"]
print(merged[display_cols].to_string(index=False))
print()
all_match_2 = merged[[c for c in merged.columns if c.startswith("match_")]].all().all()
print(f"\n{'✓' if all_match_2 else '✗'} All Phase 4B Table 2 values {'match' if all_match_2 else 'MISMATCH'} the paper.")

## 5. Reproduce Phase 4C Table 3: The four-way comparison

From the paper (Section 8.2). This is the headline result of the methodological self-correction:

In [ ]:
PAPER_TABLE_3 = pd.DataFrame({
    "Condition":     ["AC_Topicless", "AC_Topical", "FM_Topicless", "FM_Topical"],
    "phi_c_paper":   [4.08, 8.66, 9.11, 9.36],
    "phi_s_paper":   [2.96, 8.26, 8.81, 9.20],
    "drag_paper":    [6.71, 1.96, 0.85, 0.28],
    "boil_paper":    [6.12, 1.26, 0.30, 0.10],
})

summary_4c = pd.read_csv("phase-4c/06_Phase4C_Condition_Summary_VALID.csv")
anchors = summary_4c[summary_4c["Condition"].isin(PAPER_TABLE_3["Condition"])].copy()
anchors = anchors[["Condition", "phi_content_mean", "phi_specificity_mean",
                   "safety_drag_mean", "boilerplate_intensity_mean"]]
anchors.columns = ["Condition", "phi_c_csv", "phi_s_csv", "drag_csv", "boil_csv"]
anchors[["phi_c_csv", "phi_s_csv", "drag_csv", "boil_csv"]] = anchors[
    ["phi_c_csv", "phi_s_csv", "drag_csv", "boil_csv"]].round(2)

merged_3 = PAPER_TABLE_3.merge(anchors, on="Condition")
for col in ["phi_c", "phi_s", "drag", "boil"]:
    merged_3[f"match_{col}"] = (merged_3[f"{col}_paper"] - merged_3[f"{col}_csv"]).abs() < 0.01

print("PHASE 4C TABLE 3: Four-way comparison\n")
print(merged_3[["Condition", "phi_c_csv", "phi_s_csv", "drag_csv", "boil_csv"]].to_string(index=False))
print()
all_match_3 = merged_3[[c for c in merged_3.columns if c.startswith("match_")]].all().all()
print(f"\n{'✓' if all_match_3 else '✗'} All Phase 4C Table 3 values {'match' if all_match_3 else 'MISMATCH'} the paper.")
print()
print("Headline read: AC_Topicless reproduces Phase 4B's signal (phi_c=4.08).")
print("AC_Topical (same constraint stack with a real referent) produces phi_c=8.66.")
print("The compression effect Phase 4B reported was the underspecification, not the constraints.")

## 6. Reproduce Phase 4C Table 4: AC factorial decomposition

In [ ]:
PAPER_TABLE_4 = pd.DataFrame({
    "Condition": ["AC_Direct", "AC_AvoidControversy", "AC_InstitutionalTrust",
                  "AC_NoOffense", "AC_Compound_AvoidPlusTrust", "AC_Topical",
                  "AC_FictionalDisplacement", "AC_Topicless"],
    "phi_c_paper":   [9.22, 9.09, 9.03, 9.00, 8.85, 8.66, 9.27, 4.08],
    "drag_paper":    [0.48, 0.72, 1.10, 1.02, 1.62, 1.96, 0.31, 6.71],
    "boil_paper":    [0.28, 0.63, 0.64, 0.81, 1.02, 1.26, 0.14, 6.12],
})

ac_summary = summary_4c[summary_4c["Condition"].isin(PAPER_TABLE_4["Condition"])].copy()
ac_summary = ac_summary[["Condition", "phi_content_mean", "safety_drag_mean", "boilerplate_intensity_mean"]]
ac_summary.columns = ["Condition", "phi_c_csv", "drag_csv", "boil_csv"]
ac_summary[["phi_c_csv", "drag_csv", "boil_csv"]] = ac_summary[["phi_c_csv", "drag_csv", "boil_csv"]].round(2)

merged_4 = PAPER_TABLE_4.merge(ac_summary, on="Condition")
for col in ["phi_c", "drag", "boil"]:
    merged_4[f"match_{col}"] = (merged_4[f"{col}_paper"] - merged_4[f"{col}_csv"]).abs() < 0.01

print("PHASE 4C TABLE 4: AC factorial decomposition\n")
print(merged_4[["Condition", "phi_c_csv", "drag_csv", "boil_csv"]].to_string(index=False))
print()
all_match_4 = merged_4[[c for c in merged_4.columns if c.startswith("match_")]].all().all()
print(f"\n{'✓' if all_match_4 else '✗'} All Phase 4C Table 4 values {'match' if all_match_4 else 'MISMATCH'} the paper.")

## 7. Reproduce Phase 4C Table 5: Per-model behavior under topic-less vs topical

This is where the Claude Opus anomaly survives the recontextualization:

In [ ]:
PAPER_TABLE_5 = pd.DataFrame({
    "Model": ["anthropic/claude-opus-4.6", "google/gemini-3.1-pro-preview",
              "meta-llama/llama-3.3-70b-instruct", "openai/gpt-5.2", "x-ai/grok-4.20"],
    "topicless_phi_paper":   [7.78, 2.30, 3.88, 1.58, 4.90],
    "topical_phi_paper":     [9.41, 8.88, 6.62, 9.28, 9.10],
    "topicless_drag_paper":  [3.48, 7.85, 8.15, 6.60, 7.45],
})

model_cond = pd.read_csv("phase4c/07_Phase4C_Model_Condition_Summary_VALID.csv")
tl = (model_cond[model_cond["Condition"] == "AC_Topicless"]
      [["Model", "phi_content_mean", "safety_drag_mean"]]
      .rename(columns={"phi_content_mean": "topicless_phi_csv",
                       "safety_drag_mean": "topicless_drag_csv"}))
tp = (model_cond[model_cond["Condition"] == "AC_Topical"]
      [["Model", "phi_content_mean"]]
      .rename(columns={"phi_content_mean": "topical_phi_csv"}))
merged_5 = PAPER_TABLE_5.merge(tl, on="Model").merge(tp, on="Model")
for col in ["topicless_phi_csv", "topical_phi_csv", "topicless_drag_csv"]:
    merged_5[col] = merged_5[col].round(2)
for label in ["topicless_phi", "topical_phi", "topicless_drag"]:
    merged_5[f"match_{label}"] = (
        (merged_5[f"{label}_paper"] - merged_5[f"{label}_csv"]).abs() < 0.01
    )
merged_5["delta_phi"] = (merged_5["topical_phi_csv"] - merged_5["topicless_phi_csv"]).round(2)

print("PHASE 4C TABLE 5: Per-model AC topic-less vs topical\n")
print(merged_5[["Model", "topicless_phi_csv", "topical_phi_csv", "delta_phi", "topicless_drag_csv"]].to_string(index=False))
print()
all_match_5 = merged_5[[c for c in merged_5.columns if c.startswith("match_")]].all().all()
print(f"\n{'✓' if all_match_5 else '✗'} All Phase 4C Table 5 values {'match' if all_match_5 else 'MISMATCH'} the paper.")
print()
print("Claude Opus retains 7.78 phi_content under topic-less constraint pressure.")
print("GPT-5.2 collapses to 1.58. The anomaly survives the recontextualization.")

## 8. Reproduce Phase 4C Table 6: Self_Audit pair aggregate scores

In [ ]:
PAPER_TABLE_6 = pd.DataFrame({
    "Condition": ["Self_Audit_With_Context", "Self_Audit_Context_Void"],
    "phi_c_paper":      [8.57, 5.80],
    "self_audit_paper": [5.46, 7.38],
    "drag_paper":       [1.95, 1.79],
})

audit = summary_4c[summary_4c["Condition"].isin(PAPER_TABLE_6["Condition"])].copy()
audit = audit[["Condition", "phi_content_mean", "self_audit_mean", "safety_drag_mean"]]
audit.columns = ["Condition", "phi_c_csv", "self_audit_csv", "drag_csv"]
audit[["phi_c_csv", "self_audit_csv", "drag_csv"]] = audit[["phi_c_csv", "self_audit_csv", "drag_csv"]].round(2)

merged_6 = PAPER_TABLE_6.merge(audit, on="Condition")
for col in ["phi_c", "self_audit", "drag"]:
    merged_6[f"match_{col}"] = (merged_6[f"{col}_paper"] - merged_6[f"{col}_csv"]).abs() < 0.01

print("PHASE 4C TABLE 6: Self_Audit pair\n")
print(merged_6[["Condition", "phi_c_csv", "self_audit_csv", "drag_csv"]].to_string(index=False))
print()
all_match_6 = merged_6[[c for c in merged_6.columns if c.startswith("match_")]].all().all()
print(f"\n{'✓' if all_match_6 else '✗'} All Phase 4C Table 6 values {'match' if all_match_6 else 'MISMATCH'} the paper.")

## 9. Reproduce Phase 4C Table 7: Void categorical breakdown

The cleanest single finding in the paper. Grok 4.20 hallucinates a prior turn 11 of 20 trials in the context-void self-audit condition; all four other model families acknowledge the absence cleanly.

In [ ]:
PAPER_TABLE_7 = pd.DataFrame({
    "Model": ["anthropic/claude-opus-4.6", "google/gemini-3.1-pro-preview",
              "meta-llama/llama-3.3-70b-instruct", "openai/gpt-5.2",
              "x-ai/grok-4.20", "ALL_MODELS"],
    "acknowledged_paper": [20, 19, 20, 19,  9, 87],
    "hallucinated_paper": [ 0,  1,  0,  0, 11, 12],
    "deflected_paper":    [ 0,  0,  0,  1,  0,  1],
    "refused_paper":      [ 0,  0,  0,  0,  0,  0],
})

void_csv = pd.read_csv("phase4c/08_Phase4C_Void_Response_Categorical.csv")
merged_7 = PAPER_TABLE_7.merge(
    void_csv[["Model", "acknowledged", "hallucinated", "deflected", "refused"]],
    on="Model",
)
merged_7.rename(columns={
    "acknowledged": "acknowledged_csv",
    "hallucinated": "hallucinated_csv",
    "deflected": "deflected_csv",
    "refused": "refused_csv",
}, inplace=True)
for col in ["acknowledged", "hallucinated", "deflected", "refused"]:
    merged_7[f"match_{col}"] = (merged_7[f"{col}_paper"] == merged_7[f"{col}_csv"])

print("PHASE 4C TABLE 7: Self_Audit_Context_Void categorical breakdown\n")
print(merged_7[["Model", "acknowledged_csv", "hallucinated_csv", "deflected_csv", "refused_csv"]].to_string(index=False))
print()
all_match_7 = merged_7[[c for c in merged_7.columns if c.startswith("match_")]].all().all()
print(f"\n{'✓' if all_match_7 else '✗'} All Phase 4C Table 7 values {'match' if all_match_7 else 'MISMATCH'} the paper.")
print()
print("Grok 4.20 hallucinates 11/20 = 55%. Other models hallucinate at most once.")
print("This is a model-specific behavioral signature, not within sampling variance.")

## 10. Final summary

If every cell above ran without `✗` markers, the paper's tables reproduce exactly from the released CSVs.

In [ ]:
checks = {
    "Phase 4B Table 1 (vector summary)": all_match,
    "Phase 4B Table 2 (AC vs FM by model)": all_match_2,
    "Phase 4C Table 3 (four-way anchors)": all_match_3,
    "Phase 4C Table 4 (AC factorial)": all_match_4,
    "Phase 4C Table 5 (per-model topic-less vs topical)": all_match_5,
    "Phase 4C Table 6 (Self_Audit pair)": all_match_6,
    "Phase 4C Table 7 (void categorical)": all_match_7,
}

print("Final reproduction summary:\n")
for name, ok in checks.items():
    mark = "✓" if ok else "✗"
    print(f"  {mark} {name}")
print()
if all(checks.values()):
    print("All seven paper tables reproduce exactly from the released CSVs.")
    print("Reproduction verification complete.")
else:
    print("At least one table did not reproduce. Investigate above.")

## What to do with this notebook

If you have run this end-to-end and every check passed, you have verified that the paper's reported numbers are consistent with the released raw data. This is the lowest-friction form of replication available and is the first step before any deeper engagement.

The next steps in the paper's recommended replication ladder (Section 7.2):

1. **Multi-evaluator replication.** Re-score a subset of the released raw substrate outputs with at least one alternate LLM judge and a small panel of blinded human raters. Report evaluator disagreement as signal.
2. **Topic-sensitivity audit.** Re-run the four-way comparison (AC_Topicless / AC_Topical / FM_Topicless / FM_Topical) on at least one alternative topic domain (immigration, housing, climate). Cross-topic variation in the per-model drag pattern is itself informative.
3. **Content analysis of high-drag responses.** The released JSONL files (`02_GDA_Raw_VALID_canonical.jsonl` and `phase4c/02_Phase4C_Raw_VALID_canonical.jsonl`) contain the raw substrate outputs. Qualitative analysis of the AC_Topicless responses across the five models would tighten or loosen any structural claim about what the underspecification-induced compression is the linguistic phenotype of.

If you find issues, want to extend the work, or have questions, the paper is at: https://github.com/devinendorphin/alignment-friction-gda